# One Analysis, Two Engines

You are about to answer one set of business questions **twice** — once in **SQL**
and once in **pandas** — against a single small retail database. A grader checks
that each answer matches a hidden reference *and that your two engines agree with
each other*. Two tools that disagree is not a tie; it means one of them has a bug.

This mirrors Assignment 1 ("Four Formats, One Analysis"). There the analysis
travelled through four **file formats**; here it travels through two **query
engines**. Same lesson, one level up: *the tool changes, the truth does not.*

| The SQL you'll write | The pandas twin from Lesson 2 |
|---|---|
| `JOIN ... ON` (INNER / LEFT) | `df.merge(..., how=...)` |
| `LEFT JOIN ... WHERE key IS NULL` | `merge(indicator=True)`, keep `left_only` |
| `CASE WHEN ... END` | `pd.cut` / boolean masks |
| `GROUP BY` + `SUM/COUNT` | `groupby().agg()` |
| `WITH cte AS (...)` | naming an intermediate DataFrame |
| `ROW_NUMBER() OVER (PARTITION BY ...)` | `sort_values` + `groupby().cumcount()` |
| `SUM(...) OVER ()` | a groupby total divided back in |

**How grading works:** after each task run its ✅ cell — it checks *both* engines
and prints pass/fail with a pointer, never the answer. Re-running a check
overwrites its result, so iterate freely. `grader.summary()` shows your score
(14 checks). Stuck? Open `hints.md`.

> ⚠️ **One cleaning rule for the whole assignment.** The `transactions` table has
> exactly one duplicated row. Part 1 removes it into a 60-row working set, and
> **every business question uses that de-duplicated data**. In SQL:
> `WITH tx AS (SELECT DISTINCT * FROM transactions)`. In pandas:
> `transactions.drop_duplicates()`. Forget it and your totals double-count.

**Estimated time:** 2.5–3.5 hours.

In [ ]:
import sqlite3
import time
from pathlib import Path

import pandas as pd

assert Path("utils/test.py").exists(), "Run this notebook from sql-two-engines-assignment/"

from utils import data_setup
from utils import test as grader

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 12)
print("Setup complete — pandas", pd.__version__, "| SQLite via the stdlib sqlite3 module")

## Part 0 — Build the database and connect

`utils/data_setup.py` builds `data/retail.sqlite` from the pinned extracts in
`data_source/` plus two documented teaching fixtures (two no-order customers, one
duplicate transaction line). It is deterministic and needs no network — read it
later, it is short. `data/` is git-ignored; delete it and rerun this cell to
rebuild from scratch.

In [ ]:
data_setup.build_database("data/retail.sqlite")
con = sqlite3.connect("data/retail.sqlite")

# Peek at the raw tables before trusting them (the Lesson 2 habit).
print(pd.read_sql("SELECT * FROM customers LIMIT 3", con))
print()
print("transactions rows :", pd.read_sql("SELECT COUNT(*) AS n FROM transactions", con)["n"][0])
print("distinct tx ids   :", pd.read_sql("SELECT COUNT(DISTINCT transaction_id) AS n FROM transactions", con)["n"][0])

In [ ]:
grader.check_0_database(con)

## Part 1 — The one cleaning step

The `transactions` table has **61 rows but only 60 distinct `transaction_id`s** —
one line was recorded twice. Build the de-duplicated 60-row working set now and
reuse it for every question below. Keep every column; you are dropping a
duplicate *row*, not a column.

- **pandas:** `raw_tx.drop_duplicates()`
- **SQL (for the query cells later):** `WITH tx AS (SELECT DISTINCT * FROM transactions)`

In [ ]:
customers = pd.read_sql("SELECT * FROM customers", con)
raw_tx = pd.read_sql("SELECT * FROM transactions", con)

# TODO: build `tx` — the 60-row de-duplicated working set — and add a
# `revenue` column equal to quantity * unit_price. Reuse `tx` everywhere below.
tx = ...

In [ ]:
grader.check_1_dedup(tx)

## Part 2 — Six business questions, two engines each

For each question: write the **SQL** into `qN_sql` (a `pd.read_sql(...)` call) and
the **pandas** into `qN_pandas`, producing a tidy DataFrame with the exact
columns named in the comment. Row order and index never matter — the grader sorts
and rounds to 2 dp. Get the SQL right first (it reads like the question), then make
pandas produce the same table.

### Q1 — Revenue by country  ·  INNER JOIN

Join the (de-duplicated) transactions to `customers`, then total
`quantity * unit_price` per country. Only countries that actually have
transactions appear — that is what INNER means.

In [ ]:
# Output columns (both engines): ['country', 'revenue']

# TODO SQL: WITH tx AS (SELECT DISTINCT * FROM transactions)
#           join to customers, GROUP BY country, SUM(quantity*unit_price) AS revenue
q1_sql = ...        # = pd.read_sql(""" ... """, con)

# TODO pandas: tx.merge(customers, on="customer_id") then groupby("country")
q1_pandas = ...

In [ ]:
grader.check_q1(q1_sql, q1_pandas)

### Q2 — Customers with no transactions  ·  LEFT JOIN + IS NULL

Keep **every** customer, attach their transactions, and report the ones where no
transaction matched. This is the canonical interview phrasing of the LEFT JOIN +
`IS NULL` anti-join. (Two authored fixtures live exactly here.)

In [ ]:
# Output columns (both engines): ['customer_id']

# TODO SQL: customers LEFT JOIN transactions ON customer_id, WHERE <a tx column> IS NULL
q2_sql = ...

# TODO pandas: customers.merge(tx, on="customer_id", how="left", indicator=True),
#              keep rows where _merge == "left_only"
q2_pandas = ...

In [ ]:
grader.check_q2(q2_sql, q2_pandas)

### Q3 — Customers per spend band  ·  CASE WHEN

Total each *transacting* customer's spend, then bucket the total into bands and
count customers per band:

```
high : total >= 300
mid  : 100 <= total < 300
low  : total < 100
```

In [ ]:
# Output columns (both engines): ['band', 'n_customers']

# TODO SQL: a CTE for per-customer totals, then CASE WHEN ... END AS band, COUNT(*)
q3_sql = ...

# TODO pandas: per = tx.groupby("customer_id")["revenue"].sum()
#              pd.cut(per, bins=[-inf, 100, 300, inf], labels=[...], right=False).value_counts()
q3_pandas = ...

In [ ]:
grader.check_q3(q3_sql, q3_pandas)

### Q4 — Top 3 customers by total spend

The leaderboard: total per customer, biggest first, keep three.

In [ ]:
# Output columns (both engines): ['customer_id', 'total']

# TODO SQL: GROUP BY customer_id, SUM(...), ORDER BY total DESC, LIMIT 3
q4_sql = ...

# TODO pandas: groupby total, sort descending, head(3), reset_index
q4_pandas = ...

In [ ]:
grader.check_q4(q4_sql, q4_pandas)

### Q5 — Monthly revenue

Group revenue by calendar month. The month is the first 7 characters of the date
(`YYYY-MM`): SQL `substr(transaction_date, 1, 7)`, pandas `.str.slice(0, 7)`.

In [ ]:
# Output columns (both engines): ['month', 'revenue']

# TODO SQL: GROUP BY substr(transaction_date,1,7)
q5_sql = ...

# TODO pandas: add a month column, groupby("month")["revenue"].sum()
q5_pandas = ...

In [ ]:
grader.check_q5(q5_sql, q5_pandas)

### Q6 — Top customer in each country  ·  window function

Total per (customer, country), then rank customers **within** each country and
keep the #1. This is the classic interview trap: a plain `GROUP BY country` would
give you the country total but *lose* which customer it was — a window function
ranks while **keeping every row's identity**.

In [ ]:
# Output columns (both engines): ['country', 'customer_id', 'total']

# TODO SQL: per-(customer,country) totals in a CTE, then
#   ROW_NUMBER() OVER (PARTITION BY country ORDER BY total DESC, customer_id) AS rn
#   keep rn = 1
q6_sql = ...

# TODO pandas: per-(customer,country) sum, sort by [country, revenue desc, customer_id],
#   groupby("country").cumcount()+1 == 1
q6_pandas = ...

In [ ]:
grader.check_q6(q6_sql, q6_pandas)

### Q7 — Each country's share of total revenue  ·  window aggregate

Revenue per country, and each country's percentage of the grand total — computed
**without collapsing the rows**. `SUM(revenue) OVER ()` is "the whole-table total,
glued onto every row", so you can divide each country's revenue by it in place.

In [ ]:
# Output columns (both engines): ['country', 'revenue', 'pct_of_total']
# pct_of_total = 100 * revenue / (total revenue), rounded to 2 dp.

# TODO SQL: per-country revenue in a CTE, then 100.0*revenue/SUM(revenue) OVER ()
q7_sql = ...

# TODO pandas: per-country sum, divide by its .sum()
q7_pandas = ...

In [ ]:
grader.check_q7(q7_sql, q7_pandas)

**Interpretation (write one sentence):** which country dominates revenue, and
what does its share tell you about how concentrated this business is?  ______

## Part 3 — Benchmark: filter in the database, or drag everything into pandas?

`transactions_big` has 60,000 rows. You want only the EIRE rows. Two ways:

- **sql_filter** — let SQLite filter and hand pandas only the ~6,000 matches.
- **pandas_filter** — read *all* 60,000 rows into pandas, then filter there.

Both give the same answer. Time them and see what "push the work to the database"
is worth. (At this size both are fast; the *ratio* is the lesson — it grows with
the table.)

In [ ]:
def timed(fn, k=5):
    """Return (best_seconds, result) over k runs."""
    best, result = None, None
    for _ in range(k):
        t0 = time.perf_counter()
        result = fn()
        dt = time.perf_counter() - t0
        best = dt if best is None else min(best, dt)
    return best, result

# Approach A is written for you — the database does the filtering:
def sql_filter():
    return pd.read_sql("SELECT * FROM transactions_big WHERE source_country = 'EIRE'", con)

# TODO Approach B: read the WHOLE table into pandas, THEN filter to EIRE.
def pandas_filter():
    ...

# TODO: time both with timed(), then build results_df with columns
#       ['approach', 'rows_returned', 'seconds'] — one row per approach.
#       (rows_returned = len(result); both must be 6000.)
results_df = ...
results_df

In [ ]:
grader.check_benchmark(results_df)

**Takeaway (one sentence):** why does pushing the filter into SQL win, and
what happens to that gap as the table grows from 60 thousand to 60 million rows?  ______

## Part 4 — Interview drills

Four recurring interview shapes. Each is stated as a question; answer it in both
engines. These are the patterns that come up again and again — learn to recognise
the shape and the query writes itself.

### D1 — "Find the 2nd highest customer." (Nth highest)

Rank customers by total spend, descending, and return rank 2. The generalisation
(`Nth highest salary/score/customer`) is one of the most common SQL interview
questions. Use `DENSE_RANK()` so ties share a rank.

In [ ]:
# Output columns (both engines): ['customer_id', 'total']

# TODO SQL: per-customer totals, DENSE_RANK() OVER (ORDER BY total DESC), keep rank = 2
d1_sql = ...

# TODO pandas: per.rank(method="dense", ascending=False) == 2
d1_pandas = ...

In [ ]:
grader.check_d1(d1_sql, d1_pandas)

### D2 — "Find the duplicate transactions."

Which `transaction_id` appears more than once? Use the **raw** 61-row table here —
your de-duplicated `tx` already destroyed the evidence.

In [ ]:
# Output columns (both engines): ['transaction_id']

# TODO SQL: GROUP BY transaction_id HAVING COUNT(*) > 1
d2_sql = ...

# TODO pandas: raw_tx["transaction_id"].duplicated(keep=False) -> the repeated ids
d2_pandas = ...

In [ ]:
grader.check_d2(d2_sql, d2_pandas)

### D3 — "Month-over-month revenue change." (gaps between periods)

Monthly revenue (like Q5), and each month minus the previous month. The first
month has no predecessor, so its change is NULL/NaN — leave it. SQL uses
`LAG(revenue) OVER (ORDER BY month)`; pandas uses `.shift()`.

In [ ]:
# Output columns (both engines): ['month', 'revenue', 'mom_change']

# TODO SQL: monthly revenue CTE, then revenue - LAG(revenue) OVER (ORDER BY month)
d3_sql = ...

# TODO pandas: monthly revenue, then revenue - revenue.shift()
d3_pandas = ...

In [ ]:
grader.check_d3(d3_sql, d3_pandas)

### D4 — "Longest gap between a customer's purchases." (gaps between events)

Per customer, over their **distinct** purchase dates in order, find the largest
gap in days between consecutive purchases. Customers who bought on only one day
have no gap and don't appear. SQL: `julianday(date) - julianday(LAG(date) OVER
(PARTITION BY customer_id ORDER BY date))`, then `MAX`. pandas:
`groupby("customer_id")["date"].diff().dt.days`, then `.max()`.

In [ ]:
# Output columns (both engines): ['customer_id', 'longest_gap_days']

# TODO SQL: distinct (customer_id, date), LAG over date per customer, MAX(gap) per customer
d4_sql = ...

# TODO pandas: distinct customer/date, to_datetime, sort, groupby diff().dt.days, max
d4_pandas = ...

In [ ]:
grader.check_d4(d4_sql, d4_pandas)

## Score

In [ ]:
grader.summary()

### ✅ Before you submit — self-check

- [ ] Part 0: database built, `check_0` green
- [ ] Part 1: 60-row de-duplicated `tx` built, `check_1` green
- [ ] Part 2: all six questions pass in **both** engines (Q1–Q7), interpretation written
- [ ] Part 3: benchmark table built, `check_benchmark` green, takeaway written
- [ ] Part 4: all four drills pass (D1–D4)
- [ ] `grader.summary()` shows my final score (aim for 14/14)